In [18]:
import pyspark
from pyspark.sql import SparkSession

In [19]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [20]:
df = spark.read.parquet("./data/raw/type=yellow/")

## Question 2

In [33]:
from pyspark.sql import types

yellow_schema = types.StructType([
    types.StructField("VendorID", types.LongType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.LongType(), True),
    types.StructField("DOLocationID", types.LongType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [30]:
import os

if not os.path.isdir("./data/pq/"):
    os.makedirs("./data/pq/")

In [34]:
year = 2025

for month in range(1, 13):

    input_path = f'data/raw/type=yellow/year={year}/month={month:02d}/'
    if not os.path.isdir(input_path):
        continue

    df_yellow = spark.read \
        .schema(yellow_schema) \
        .parquet(input_path)

    if df_yellow.count() == 0:
        continue

    print(f'processing data for {year}/{month}')
    output_path = f'data/pq/type=yellow/year={year}/month={month:02d}/'
    df_yellow \
        .repartition(4) \
        .write.parquet(output_path)

processing data for 2025/11


## Question 3

In [ ]:
input_path = f'data/pq/type=yellow/'

df_yellow = spark.read \
    .schema(yellow_schema) \
    .parquet(input_path)

df_yellow.createOrReplaceTempView("yellow_trips")

In [36]:
trip_count = spark.sql("""
    SELECT COUNT(*) as trip_count
    FROM yellow_trips
    WHERE 1 = 1
    AND DAY(tpep_pickup_datetime) = 15 
    AND MONTH(tpep_pickup_datetime) = 11
    AND YEAR(tpep_pickup_datetime) = 2025
""").collect()

print(f"Taxi trips on November 15th: {trip_count}")


Taxi trips on November 15th: [Row(trip_count=166857)]


## Question 4

In [39]:
max_duration = df_yellow.agg(
    F.max(
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
    ).alias("max_hours")
).collect()

print(f"Longest trip duration: {max_duration} hours")

Longest trip duration: [Row(max_hours=90.64666666666666)] hours


## Question 5

In [40]:
!wget -O data/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://d37ci6vzurychx.cloudfr]87Saving 'data/taxi_zone_lookup.csv'
87data/taxi_zone_looku 100% [=============================>]    3.01K    --.-KB/s87HTTP response 200  [https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv]
87data/taxi_zone_looku 100% [=============================>]    3.01K    --.-KB/s87[Files: 1  Bytes: 3.01K [4.62KB]8

In [41]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('data/taxi_zone_lookup.csv')

In [43]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [46]:
from pyspark.sql import functions as F

# Join yellow trips with zone lookup on pickup location
df_pickup_zones = df_yellow.join(
    df_zones,
    df_yellow.PULocationID == df_zones.LocationID,
    "left"
)

# Count trips by zone and sort to find the least frequent
least_frequent_zone = df_pickup_zones\
    .groupBy("Zone")\
    .count() \
    .orderBy(F.col("Zone").asc()) \
    .orderBy(F.col("count").asc()) \
    .limit(10)

least_frequent_zone.show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+

